# Blog Writing Evaluation — LangSmith Lab (20-Example Run)

**Domain:** AI-assisted blog writing  
**LangSmith project:** `blog_eval_langsmith`  
**What this notebook does:**
1. Traces a live blog-generation call with `@traceable`.
2. Creates a **20-example** LangSmith dataset (`blog-writing-eval-20`).
3. Runs an evaluation experiment with three LLM-as-judge evaluators.
4. Compares `gpt-4o-mini` vs `gpt-4o` side-by-side (A/B).
5. Summarises scores and draws a plain-English conclusion.
6. Demonstrates reproducibility at `temperature=0.7`.
7. **Compares 10-example vs 20-example results** to check whether conclusions hold at larger scale.

In [ ]:
%pip install langsmith langchain-openai openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "blog_eval_langsmith"

print("Project :", os.environ["LANGSMITH_PROJECT"])
print("Endpoint:", os.environ["LANGSMITH_ENDPOINT"])
print("API key set:", bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")))
print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
import json
import re
from statistics import mean
from typing import Any, Dict, List, Optional

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langsmith import Client, traceable
from langsmith.evaluation import EvaluationResult

client = Client()
print("LangSmith client ready.")

## 1. Tracing

`@traceable` logs every call to LangSmith automatically.  
Every blog post generated below will appear in the **blog_eval_langsmith** project.

In [ ]:
@traceable(name="generate-blog")
def generate_blog(inputs: Dict[str, Any], model_name: str = "gpt-4o-mini") -> Dict[str, str]:
    """Generate a blog post from topic + brief. Traced automatically by LangSmith."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert blog writer. Write a 600-800 word blog post that is clear, "
         "useful, and easy to scan. Use headings, short paragraphs, and a natural human tone. "
         "Do not mention that you are an AI. Stay faithful to the audience, tone, and goal in the brief."),
        ("user", "Topic: {topic}\n\nBrief: {brief}\n\nWrite the blog post now."),
    ])
    chain = prompt | ChatOpenAI(model=model_name, temperature=0.7)
    result = chain.invoke({"topic": inputs["topic"], "brief": inputs["brief"]})
    return {"blog_post": getattr(result, "content", str(result)).strip()}


# Quick smoke-test — this trace will appear in LangSmith
sample = generate_blog({
    "topic": "How AI can help small businesses with marketing",
    "brief": "Audience: non-technical small business owners. Tone: friendly. Goal: explain 3 practical AI marketing tips.",
})
print(sample["blog_post"][:800], "\n...")

## 2. Dataset

20 blog-writing examples (original 10 + 10 new topics).  
Each has a `topic` + `brief` as inputs and a reference `outline` +  
`key_requirements` as outputs. We push them to LangSmith as `blog-writing-eval-20`.

In [ ]:
DATASET_NAME = "blog-writing-eval-20"

BLOG_EXAMPLES = [
    {
        "id": "blog_001",
        "inputs": {
            "topic": "How AI can help small businesses with marketing",
            "brief": "Audience: non-technical small business owners. Tone: friendly, encouraging, practical. Goal: explain 3-4 practical ways AI can support marketing without jargon.",
        },
        "outputs": {
            "outline": ["Intro: marketing overwhelm for small businesses", "Section 1: AI for idea generation", "Section 2: AI for drafting emails and captions", "Section 3: AI for audience insights"],
            "key_requirements": ["Mention at least 3 distinct AI use cases", "Use simple, non-technical language", "Use headings and short paragraphs"],
        },
    },
    {
        "id": "blog_002",
        "inputs": {
            "topic": "Using AI to overcome writer's block",
            "brief": "Audience: solo creators. Tone: honest and encouraging. Goal: show how AI can help start drafts without replacing their voice.",
        },
        "outputs": {
            "outline": ["Intro: what writer's block feels like", "Section 1: AI for brainstorming angles", "Section 2: AI for rough outlines and first drafts", "Section 3: keeping your voice"],
            "key_requirements": ["Explain that AI does not replace the creator's voice", "Give at least 2 concrete prompts they can try", "Tone should be supportive, not hype-heavy"],
        },
    },
    {
        "id": "blog_003",
        "inputs": {
            "topic": "Planning a month of content with AI",
            "brief": "Audience: small business owners. Tone: practical and step-by-step. Goal: explain how to use AI to create a simple 30-day content plan.",
        },
        "outputs": {
            "outline": ["Intro: why planning ahead helps", "Section 1: defining your main themes", "Section 2: using AI to generate a list of ideas", "Section 3: turning ideas into a calendar"],
            "key_requirements": ["Describe a clear sequence of steps", "Mention using AI to generate a batch of ideas", "Include the idea of reviewing and editing the plan"],
        },
    },
    {
        "id": "blog_004",
        "inputs": {
            "topic": "Repurposing one blog post into multiple formats with AI",
            "brief": "Audience: busy professionals. Tone: efficient and time-saving. Goal: show how to turn one blog into social posts, emails, and short scripts.",
        },
        "outputs": {
            "outline": ["Intro: why repurposing saves time", "Section 1: start with one pillar blog post", "Section 2: social media snippets", "Section 3: email newsletters", "Section 4: short scripts"],
            "key_requirements": ["Mention at least 3 formats", "Explain that all pieces should align with one core message", "Tone should focus on saving time and energy"],
        },
    },
    {
        "id": "blog_005",
        "inputs": {
            "topic": "Common mistakes when using AI for content",
            "brief": "Audience: creators who already tried AI. Tone: honest, slightly cautious, constructive. Goal: highlight 3-5 mistakes and how to avoid them.",
        },
        "outputs": {
            "outline": ["Intro: AI hype vs. reality", "Section 1: copying AI output without editing", "Section 2: ignoring your audience's voice", "Section 3: over-automating and losing personality"],
            "key_requirements": ["List at least 3 specific mistakes", "Include suggestions for how to avoid each mistake", "Tone should be constructive, not shaming"],
        },
    },
    {
        "id": "blog_006",
        "inputs": {
            "topic": "Using AI to research a blog topic before writing",
            "brief": "Audience: beginner bloggers. Tone: clear and educational. Goal: show how to gather ideas, questions, and source leads before drafting.",
        },
        "outputs": {
            "outline": ["Intro: research helps you write with confidence", "Section 1: ask AI for topic angles and key questions", "Section 2: use AI to organise research notes", "Section 3: verify and cite your sources"],
            "key_requirements": ["Explain that AI research should be verified", "Give at least one example of a useful research prompt", "Keep the advice beginner-friendly"],
        },
    },
    {
        "id": "blog_007",
        "inputs": {
            "topic": "Writing a product launch post with AI",
            "brief": "Audience: startup founders. Tone: persuasive but grounded. Goal: explain how AI can help draft a launch announcement without sounding robotic.",
        },
        "outputs": {
            "outline": ["Intro: what makes a launch post effective", "Section 1: define the value proposition", "Section 2: AI for headline and hook ideas", "Section 3: AI to structure the body"],
            "key_requirements": ["Mention value proposition and audience pain points", "Highlight the need to edit for brand voice", "Keep the tone persuasive, not overhyped"],
        },
    },
    {
        "id": "blog_008",
        "inputs": {
            "topic": "Building a content calendar for a local business with AI",
            "brief": "Audience: local shop owners. Tone: practical and friendly. Goal: explain how to make a simple weekly or monthly calendar.",
        },
        "outputs": {
            "outline": ["Intro: why local businesses benefit from planning", "Section 1: choose themes around products and events", "Section 2: use AI to propose posts", "Section 3: review and adjust"],
            "key_requirements": ["Tie the calendar to real local business goals", "Suggest a simple repeatable planning routine", "Use a friendly, encouraging tone"],
        },
    },
    {
        "id": "blog_009",
        "inputs": {
            "topic": "Creating evergreen tutorials with AI",
            "brief": "Audience: content marketers. Tone: strategic and practical. Goal: show how AI can help draft timeless tutorials that stay useful over time.",
        },
        "outputs": {
            "outline": ["Intro: what evergreen content is", "Section 1: pick topics that solve durable problems", "Section 2: use AI to outline step-by-step tutorials", "Section 3: optimize for search"],
            "key_requirements": ["Explain why evergreen content lasts", "Include a reminder to update and refresh content", "Keep the advice practical for marketers"],
        },
    },
    {
        "id": "blog_010",
        "inputs": {
            "topic": "Maintaining brand voice while using AI for content",
            "brief": "Audience: marketing teams. Tone: confident and precise. Goal: show how to keep content consistent with brand guidelines while using AI speed.",
        },
        "outputs": {
            "outline": ["Intro: speed is useful, but voice matters", "Section 1: define the brand's voice and style rules", "Section 2: use AI with clear examples and constraints", "Section 3: human review process"],
            "key_requirements": ["Mention brand guidelines or style rules", "Explain the importance of human review", "Keep the tone professional and practical"],
        },
    },
    {
        "id": "blog_011",
        "inputs": {
            "topic": "Writing SEO-friendly blog posts with AI",
            "brief": "Audience: bloggers and small business owners new to SEO. Tone: practical and demystifying. Goal: explain how to use AI to write posts that rank.",
        },
        "outputs": {
            "outline": ["Intro: why SEO matters for blog reach", "Section 1: use AI to find the right keywords", "Section 2: AI for writing meta titles and descriptions", "Section 3: structure for readability"],
            "key_requirements": ["Explain keyword research in plain terms", "Give at least one example of an AI SEO prompt", "Mention on-page structure (headings, links)"],
        },
    },
    {
        "id": "blog_012",
        "inputs": {
            "topic": "Using AI to write compelling case studies",
            "brief": "Audience: B2B marketers and sales teams. Tone: structured and professional. Goal: show how AI speeds up case study writing without losing authenticity.",
        },
        "outputs": {
            "outline": ["Intro: why case studies convert", "Section 1: gather the raw story from the client", "Section 2: use AI to structure the problem-solution-result arc", "Section 3: editing for accuracy"],
            "key_requirements": ["Describe the problem-solution-result structure", "Stress that the client story must come first", "Mention how to handle client quotes and data"],
        },
    },
    {
        "id": "blog_013",
        "inputs": {
            "topic": "Creating FAQ pages that rank on Google with AI",
            "brief": "Audience: website owners and e-commerce teams. Tone: clear and practical. Goal: explain how to use AI to identify real customer questions and write answers.",
        },
        "outputs": {
            "outline": ["Intro: FAQ pages as SEO and trust tools", "Section 1: use AI to generate a list of real customer questions", "Section 2: write concise, direct answers", "Section 3: optimize for featured snippets"],
            "key_requirements": ["Mention the connection between FAQs and featured snippets", "Explain how to surface real customer questions using AI", "Keep answers short and structured"],
        },
    },
    {
        "id": "blog_014",
        "inputs": {
            "topic": "Writing email newsletters with AI",
            "brief": "Audience: founders and creators with small email lists. Tone: friendly and action-oriented. Goal: show how AI can help write newsletters that people read.",
        },
        "outputs": {
            "outline": ["Intro: why newsletters still work", "Section 1: AI for subject lines and preview text", "Section 2: AI to draft the body in your voice", "Section 3: writing a strong call to action"],
            "key_requirements": ["Include subject line and preview text as a distinct section", "Explain how to keep the AI output sounding personal", "Mention the call to action"],
        },
    },
    {
        "id": "blog_015",
        "inputs": {
            "topic": "Using AI to draft thought leadership articles",
            "brief": "Audience: executives and senior professionals. Tone: authoritative and reflective. Goal: explain how leaders can use AI to turn their ideas into polished articles.",
        },
        "outputs": {
            "outline": ["Intro: thought leadership requires a real point of view", "Section 1: start with your own insight, not AI", "Section 2: use AI to structure and expand the argument", "Section 3: editing for voice and impact"],
            "key_requirements": ["Stress that the original idea must come from the human", "Explain the structure of a thought leadership piece", "Mention editing for personal voice"],
        },
    },
    {
        "id": "blog_016",
        "inputs": {
            "topic": "How to use AI for product description writing",
            "brief": "Audience: e-commerce store owners. Tone: practical and results-focused. Goal: show how to write product descriptions that sell using AI.",
        },
        "outputs": {
            "outline": ["Intro: most product descriptions are forgettable", "Section 1: give AI the right product details and audience", "Section 2: focus on benefits, not just features", "Section 3: A/B test variants"],
            "key_requirements": ["Distinguish between features and benefits", "Explain how to prompt AI with product details", "Mention A/B testing variants"],
        },
    },
    {
        "id": "blog_017",
        "inputs": {
            "topic": "Creating step-by-step how-to guides with AI",
            "brief": "Audience: educators, coaches, and content creators. Tone: clear and instructional. Goal: explain how to use AI to structure and write guides that are easy to follow.",
        },
        "outputs": {
            "outline": ["Intro: how-to guides build authority and trust", "Section 1: define the outcome and audience before prompting AI", "Section 2: use AI to draft numbered steps", "Section 3: add examples and edge cases"],
            "key_requirements": ["Explain how to define scope before using AI", "Describe how to structure numbered steps", "Mention adding examples and edge cases"],
        },
    },
    {
        "id": "blog_018",
        "inputs": {
            "topic": "Using AI to write your brand origin story",
            "brief": "Audience: entrepreneurs and small business founders. Tone: personal and narrative. Goal: help founders use AI to shape their story into something compelling.",
        },
        "outputs": {
            "outline": ["Intro: origin stories build emotional connection", "Section 1: gather the raw facts and feelings before using AI", "Section 2: use AI to structure a narrative arc", "Section 3: editing for authenticity"],
            "key_requirements": ["Stress that raw material must come from the founder", "Describe a simple narrative arc (problem to pivot to purpose)", "Explain the editing step to preserve authenticity"],
        },
    },
    {
        "id": "blog_019",
        "inputs": {
            "topic": "AI writing tools for non-native English speakers",
            "brief": "Audience: professionals and creators whose first language is not English. Tone: empathetic and empowering. Goal: show how AI can reduce the language barrier.",
        },
        "outputs": {
            "outline": ["Intro: language should not limit great ideas", "Section 1: AI for grammar correction and clarity", "Section 2: using AI to expand short ideas into full paragraphs", "Section 3: preserving cultural voice"],
            "key_requirements": ["Address the fear of language being a barrier", "Explain grammar and expansion use cases", "Include a note about preserving cultural perspective"],
        },
    },
    {
        "id": "blog_020",
        "inputs": {
            "topic": "Writing community and culture posts with AI",
            "brief": "Audience: community managers and HR teams. Tone: warm and inclusive. Goal: show how AI can help write internal culture posts and community updates.",
        },
        "outputs": {
            "outline": ["Intro: culture content builds belonging", "Section 1: team spotlights and how AI helps draft them", "Section 2: using AI for event recaps and announcements", "Section 3: maintaining inclusive tone"],
            "key_requirements": ["Cover at least 2 content types (spotlight, recap, or announcement)", "Stress the importance of avoiding corporate-speak", "Mention how to inject specific details and names"],
        },
    },
]

print(f"Defined {len(BLOG_EXAMPLES)} blog examples.")

In [ ]:
existing = [d.name for d in client.list_datasets()]

if DATASET_NAME not in existing:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="20 blog-writing examples: topic + brief → reference outline + key requirements",
    )
    client.create_examples(
        inputs    = [e["inputs"]  for e in BLOG_EXAMPLES],
        outputs   = [e["outputs"] for e in BLOG_EXAMPLES],
        dataset_id = dataset.id,
    )
    print(f"Created dataset '{DATASET_NAME}' with {len(BLOG_EXAMPLES)} examples.")
else:
    dataset = next(d for d in client.list_datasets() if d.name == DATASET_NAME)
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload.")

print(f"Dataset id: {dataset.id}")

## 3. Run an Experiment

A **target function** maps dataset inputs → outputs.  
Three **LLM-as-judge evaluators** score each output on coverage, structure, and tone (0–100).

In [ ]:
# Target function: wraps generate_blog for gpt-4o-mini
def target_gpt4o_mini(inputs: dict) -> dict:
    return generate_blog(inputs, model_name="gpt-4o-mini")

In [ ]:
COVERAGE_PROMPT = """You are an expert evaluator of blog post coverage.

Given the inputs (topic + brief), the generated blog post, and the reference outline
and key requirements, score how well the blog covers the expected content.

Scoring (0-100):
0-20: Misses most key points.
21-50: Covers some points but omits several important elements.
51-80: Generally covers the key points with minor omissions.
81-100: Strong coverage of all outline points and key requirements.

Reply with ONLY a JSON object, no markdown, no preamble:
{"score": <integer 0-100>, "feedback": "<one sentence>"}

Inputs: PLACEHOLDER_INPUTS
Generated output: PLACEHOLDER_OUTPUTS
Reference outputs: PLACEHOLDER_REFERENCE"""

STRUCTURE_PROMPT = """You are an expert evaluator of blog post structure.

Given the inputs (topic + brief), the generated blog post, and the reference outline,
score the structure and organisation of the blog post.

Scoring (0-100):
0-20: No clear structure — missing intro, sections, or conclusion.
21-50: Some structure but significant problems.
51-80: Mostly correct structure with minor issues.
81-100: Excellent — clear intro, descriptive H2 sections, short paragraphs, concise conclusion.

Reply with ONLY a JSON object, no markdown, no preamble:
{"score": <integer 0-100>, "feedback": "<one sentence>"}

Inputs: PLACEHOLDER_INPUTS
Generated output: PLACEHOLDER_OUTPUTS
Reference outputs: PLACEHOLDER_REFERENCE"""

TONE_PROMPT = """You are an expert evaluator of blog post tone and audience fit.

Given the inputs (topic + brief), the generated blog post, and the reference,
score how well the blog matches the requested tone and addresses the audience.

Scoring (0-100):
0-20: Tone completely mismatched; audience ignored.
21-50: Tone partially matches but frequently wrong.
51-80: Generally matches tone and audience with minor lapses.
81-100: Excellent — consistent tone, audience-appropriate language throughout.

Reply with ONLY a JSON object, no markdown, no preamble:
{"score": <integer 0-100>, "feedback": "<one sentence>"}

Inputs: PLACEHOLDER_INPUTS
Generated output: PLACEHOLDER_OUTPUTS
Reference outputs: PLACEHOLDER_REFERENCE"""


def _make_evaluator(prompt_template: str, feedback_key: str, judge_model: str = "gpt-4o-mini"):
    """Return a LangSmith-compatible evaluator using explicit PLACEHOLDER substitution."""
    def evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> EvaluationResult:
        filled = (
            prompt_template
            .replace("PLACEHOLDER_INPUTS",    json.dumps(inputs,            ensure_ascii=False))
            .replace("PLACEHOLDER_OUTPUTS",   json.dumps(outputs,           ensure_ascii=False))
            .replace("PLACEHOLDER_REFERENCE", json.dumps(reference_outputs, ensure_ascii=False))
        )
        llm = ChatOpenAI(model=judge_model, temperature=0)
        content = getattr(llm.invoke(filled), "content", "").strip()
        content = re.sub(r"^```(?:json)?\s*", "", content)
        content = re.sub(r"\s*```$", "", content)
        try:
            data    = json.loads(content)
            score   = float(data["score"])
            comment = str(data.get("feedback", ""))
        except Exception as e:
            score, comment = 0.0, f"parse_error={e} | raw={content[:200]}"
        return EvaluationResult(key=feedback_key, score=score, comment=comment)
    evaluator.__name__ = feedback_key
    return evaluator


def make_evaluators(judge_model: str = "gpt-4o-mini") -> list:
    return [
        _make_evaluator(COVERAGE_PROMPT,  "coverage",          judge_model),
        _make_evaluator(STRUCTURE_PROMPT, "structure_clarity",  judge_model),
        _make_evaluator(TONE_PROMPT,      "tone_match",         judge_model),
    ]

print("Evaluators defined.")

In [ ]:
results_mini = client.evaluate(
    target_gpt4o_mini,
    data=DATASET_NAME,
    evaluators=make_evaluators(),
    experiment_prefix="gpt-4o-mini",
    max_concurrency=2,
)
print(results_mini)

## 4. Compare Two Models (A/B)

Run the same **20 examples** with `gpt-4o`.  
The LangSmith UI lets you compare both experiments side-by-side.

In [ ]:
def target_gpt4o(inputs: dict) -> dict:
    return generate_blog(inputs, model_name="gpt-4o")

results_4o = client.evaluate(
    target_gpt4o,
    data=DATASET_NAME,
    evaluators=make_evaluators(),
    experiment_prefix="gpt-4o",
    max_concurrency=2,
)
print(results_4o)

## 5. Results Summary

In [ ]:
def extract_scores(results, feedback_keys=("coverage", "structure_clarity", "tone_match")):
    """Pull per-key mean scores out of a client.evaluate() result object."""
    buckets = {k: [] for k in feedback_keys}
    rows = getattr(results, "_results", None) or getattr(results, "results", None) or []
    for row in rows:
        fb_list = (
            (row.get("evaluation_results") or {}).get("results", [])
            if isinstance(row, dict)
            else (getattr(row, "feedback", None) or [])
        )
        for fb in fb_list:
            key = getattr(fb, "key", None)
            raw = getattr(fb, "score", None) or getattr(fb, "value", None)
            if key in buckets and raw is not None:
                try:
                    buckets[key].append(float(raw))
                except (TypeError, ValueError):
                    pass
    return {
        "n": len(rows),
        **{k: round(float(mean(v)), 1) if v else 0.0 for k, v in buckets.items()},
    }


scores_mini = extract_scores(results_mini)
scores_4o   = extract_scores(results_4o)

summary = pd.DataFrame([
    {"model": "gpt-4o-mini", **scores_mini},
    {"model": "gpt-4o",      **scores_4o},
])
print(summary.to_string(index=False))

In [ ]:
# Plain-English conclusion
dims = ["coverage", "structure_clarity", "tone_match"]

for row in summary.itertuples(index=False):
    scores = {d: getattr(row, d, 0.0) for d in dims}
    strongest = max(scores, key=scores.get)
    weakest   = min(scores, key=scores.get)
    print(
        f"{row.model}: coverage={scores['coverage']:.0f}  "
        f"structure={scores['structure_clarity']:.0f}  "
        f"tone={scores['tone_match']:.0f}  "
        f"-> strongest: {strongest}, weakest: {weakest}"
    )

## 6. Reproducibility

From the project brief: same code + same seed ≠ guaranteed identical results.  
At `temperature=0.7` outputs will differ across runs — that is expected.

In [ ]:
repro_input = BLOG_EXAMPLES[0]["inputs"]

run1 = generate_blog(repro_input)
run2 = generate_blog(repro_input)

print("Run 1 word count:", len(run1["blog_post"].split()))
print("Run 2 word count:", len(run2["blog_post"].split()))
print("Identical output:", run1["blog_post"] == run2["blog_post"])
print("\nAt temperature=0.7 outputs almost always differ.")
print("Use temperature=0 if determinism is required.")

## 7. Scale Check: 10-Example vs 20-Example Results

The original run used 10 examples. This section compares those published results  
with the scores from the 20-example run above to check whether the conclusions hold at larger scale.

In [ ]:
# Published 10-example scores (from evaluation_report.md)
scores_10_mini = {"n": 10, "coverage": 80.5, "structure_clarity": 82.0, "tone_match": 81.0}
scores_10_4o   = {"n": 10, "coverage": 78.5, "structure_clarity": 83.0, "tone_match": 82.0}

dims = ["coverage", "structure_clarity", "tone_match"]

comparison = pd.DataFrame([
    {"run": "10-example", "model": "gpt-4o-mini", **{d: scores_10_mini[d] for d in dims}},
    {"run": "10-example", "model": "gpt-4o",      **{d: scores_10_4o[d]   for d in dims}},
    {"run": "20-example", "model": "gpt-4o-mini", **{d: scores_mini[d]    for d in dims}},
    {"run": "20-example", "model": "gpt-4o",      **{d: scores_4o[d]      for d in dims}},
])

comparison["avg"] = comparison[dims].mean(axis=1).round(1)
print(comparison.to_string(index=False))

In [ ]:
# Delta: how much did scores shift from 10 to 20 examples?
for model in ["gpt-4o-mini", "gpt-4o"]:
    r10 = comparison[(comparison["run"] == "10-example") & (comparison["model"] == model)].iloc[0]
    r20 = comparison[(comparison["run"] == "20-example") & (comparison["model"] == model)].iloc[0]
    deltas = {d: round(r20[d] - r10[d], 1) for d in dims}
    avg_delta = round(r20["avg"] - r10["avg"], 1)
    direction = "stable" if abs(avg_delta) <= 2 else ("improved" if avg_delta > 0 else "dropped")
    print(
        f"{model}: coverage {deltas['coverage']:+.1f}  "
        f"structure {deltas['structure_clarity']:+.1f}  "
        f"tone {deltas['tone_match']:+.1f}  avg_delta={avg_delta:+.1f}  -> {direction}"
    )